# Weird AI Lesson 6: Classification Fine-Tuning Side Quest

In this notebook, you will explore how the same transformer foundation used for text generation can be adapted for a classification task.

This lesson is a **side quest** in the Weird AI project:

- Previous lessons focused on generating lyrics.
- This lesson asks the model to classify lyric-like text.
- The goal is to understand how fine-tuning changes a model for a different task.

By the end of this notebook, you should understand:

1. How classification differs from generation
2. How labeled datasets differ from unlabeled pretraining data
3. Why sequences need to be padded or truncated
4. Why a classification head has a different output size than a generation head
5. How classification accuracy is evaluated

## Section 1: Generation vs Classification

In the Weird AI generator, the model predicts the **next token**.

```text
Input text
   ↓
Transformer
   ↓
Vocabulary output layer
   ↓
Next-token prediction
```

In classification, the model predicts a **label**.

```text
Input text
   ↓
Transformer
   ↓
Classification output layer
   ↓
Class label
```

For this notebook, we will classify short lyric-like text samples.

In [1]:
# Run this cell first.

import torch
import pandas as pd

from pathlib import Path
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

torch.manual_seed(123)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


## Section 2: Create a Small Labeled Dataset

Pretraining used unlabeled lyric text. Classification requires labels.

For this notebook, we will start with a tiny two-class dataset:

- `0` = serious lyric
- `1` = silly/comedic lyric

This is intentionally small and imperfect. The goal is to understand the workflow, not to build a production-quality classifier yet.

In [2]:
data = [
    {"text": "I walked alone beneath the rain and wondered where you were", "label": 0},
    {"text": "The moon was cold, the night was long, my heart forgot the tune", "label": 0},
    {"text": "Every road reminds me of the love I left behind", "label": 0},
    {"text": "The silence in this empty room still whispers your name", "label": 0},
    {"text": "I kept your letter folded by the window in the storm", "label": 0},
    {"text": "The stars fell softly as I waited for your call", "label": 0},

    {"text": "My sandwich ran away and joined a polka band", "label": 1},
    {"text": "I fell in love with a toaster wearing neon shoes", "label": 1},
    {"text": "The cafeteria spaghetti tried to steal my car", "label": 1},
    {"text": "My homework ate my dog and then blamed it on the cat", "label": 1},
    {"text": "The disco duck accountant audited my socks", "label": 1},
    {"text": "I wrote a love song to my broken microwave", "label": 1},
]

df = pd.DataFrame(data)

df

,text,label
0,I walked alone beneath the rain and wondered w...,0
1,"The moon was cold, the night was long, my hear...",0
2,Every road reminds me of the love I left behind,0
3,The silence in this empty room still whispers ...,0
4,I kept your letter folded by the window in the...,0
5,The stars fell softly as I waited for your call,0
6,My sandwich ran away and joined a polka band,1
7,I fell in love with a toaster wearing neon shoes,1
8,The cafeteria spaghetti tried to steal my car,1
9,My homework ate my dog and then blamed it on t...,1


## Section 3: Inspect the Dataset

Before building models, always inspect the data.

TODO:

1. Count how many examples are in each class.
2. Print a few examples from each class.
3. Explain whether this dataset is balanced.

In [3]:
print(df["label"].value_counts())

print("\nSerious examples:")
print(df[df["label"] == 0].head())

print("\nSilly examples:")
print(df[df["label"] == 1].head())

label
0    6
1    6
Name: count, dtype: int64

Serious examples:
                                                text  label
0  I walked alone beneath the rain and wondered w...      0
1  The moon was cold, the night was long, my hear...      0
2    Every road reminds me of the love I left behind      0
3  The silence in this empty room still whispers ...      0
4  I kept your letter folded by the window in the...      0

Silly examples:
                                                 text  label
6        My sandwich ran away and joined a polka band      1
7    I fell in love with a toaster wearing neon shoes      1
8       The cafeteria spaghetti tried to steal my car      1
9   My homework ate my dog and then blamed it on t...      1
10         The disco duck accountant audited my socks      1


## Section 4: Split into Training, Validation, and Test Sets

For classification, we need separate datasets:

- Training data: used to update model weights
- Validation data: used to monitor performance during development
- Test data: used for final evaluation

With a tiny dataset, the split will be unstable. In a real project, you would need many more labeled examples.

In [4]:
train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    random_state=123,
    stratify=df["label"]
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=123,
    stratify=temp_df["label"]
)

print(f"Training rows: {len(train_df)}")
print(f"Validation rows: {len(val_df)}")
print(f"Test rows: {len(test_df)}")

print("\nTraining distribution:")
print(train_df["label"].value_counts())

print("\nValidation distribution:")
print(val_df["label"].value_counts())

print("\nTest distribution:")
print(test_df["label"].value_counts())

Training rows: 8
Validation rows: 2
Test rows: 2

Training distribution:
label
1    4
0    4
Name: count, dtype: int64

Validation distribution:
label
1    1
0    1
Name: count, dtype: int64

Test distribution:
label
1    1
0    1
Name: count, dtype: int64


## Section 5: Tokenize and Pad Text

A classifier cannot directly process strings. We need to convert text into token IDs.

For this notebook, we will use the existing `SimpleCharacterTokenizer` from Weird AI.

Padding is necessary because batches require tensors with the same sequence length.

For this simple character tokenizer, we will use `0` as the padding ID.

In [5]:
from weird_ai.tokenizer import SimpleCharacterTokenizer

all_text = "\n".join(df["text"].tolist())

tokenizer = SimpleCharacterTokenizer(all_text)

print(f"Vocabulary size: {len(tokenizer.chars)}")
print(tokenizer.chars)

Vocabulary size: 30
['\n', ' ', ',', 'E', 'I', 'M', 'T', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'r', 's', 't', 'u', 'v', 'w', 'y']


In [6]:
sample_text = df.iloc[0]["text"]

encoded = tokenizer.encode(sample_text)
decoded = tokenizer.decode(encoded)

print(sample_text)
print(encoded)
print(decoded)

I walked alone beneath the rain and wondered where you were
[4, 1, 28, 7, 18, 17, 11, 10, 1, 7, 18, 21, 20, 11, 1, 8, 11, 20, 11, 7, 25, 14, 1, 25, 14, 11, 1, 23, 7, 15, 20, 1, 7, 20, 10, 1, 28, 21, 20, 10, 11, 23, 11, 10, 1, 28, 14, 11, 23, 11, 1, 29, 21, 26, 1, 28, 11, 23, 11]
I walked alone beneath the rain and wondered where you were


## Section 6: Build a Classification Dataset

This dataset should return:

```python
input_ids, label
```

where:

- `input_ids` is a padded tensor of token IDs
- `label` is the class label, either `0` or `1`

TODO:

Complete the missing parts of `LyricsClassificationDataset`.

In [13]:
class LyricsClassificationDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_length=None, pad_token_id=0):
        self.dataframe = dataframe.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.pad_token_id = pad_token_id

        self.encoded_texts = [
            tokenizer.encode(text)
            for text in self.dataframe["text"]
        ]

        if max_length is None:
            self.max_length = self._longest_encoded_length()
        else:
            self.max_length = max_length

        # Truncate sequences that are too long.
        self.encoded_texts = [
            encoded_text[:self.max_length]
            for encoded_text in self.encoded_texts
        ]

        # Pad sequences that are shorter than max_length.
        self.encoded_texts = [
            encoded_text + [self.pad_token_id] * (self.max_length - len(encoded_text))
            for encoded_text in self.encoded_texts
        ]

    def __getitem__(self, index):
        encoded = self.encoded_texts[index]
        label = self.dataframe.iloc[index]["label"]

        return (
            torch.tensor(encoded, dtype=torch.long),
            torch.tensor(label, dtype=torch.long)
        )

    def __len__(self):
        return len(self.dataframe)

    def _longest_encoded_length(self):
        max_length = 0

        for encoded_text in self.encoded_texts:
            if len(encoded_text) > max_length:
                max_length = len(encoded_text)

        return max_length

## Section 7: Create DataLoaders

DataLoaders group individual examples into batches.

TODO:

1. Create training, validation, and test datasets.
2. Create DataLoaders for each split.
3. Inspect one batch.

In [14]:
# TODO:
# After completing LyricsClassificationDataset, this cell should work.

train_dataset = LyricsClassificationDataset(train_df, tokenizer)
val_dataset = LyricsClassificationDataset(
    val_df,
    tokenizer,
    max_length=train_dataset.max_length
)
test_dataset = LyricsClassificationDataset(
    test_df,
    tokenizer,
    max_length=train_dataset.max_length
)

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=4, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=4, shuffle=False)

input_batch, label_batch = next(iter(train_loader))

print(input_batch)
print(label_batch)
print(input_batch.shape)
print(label_batch.shape)

tensor([[ 6, 14, 11,  1, 19, 21, 21, 20,  1, 28,  7, 24,  1,  9, 21, 18, 10,  2,
          1, 25, 14, 11,  1, 20, 15, 13, 14, 25,  1, 28,  7, 24,  1, 18, 21, 20,
         13,  2,  1, 19, 29,  1, 14, 11,  7, 23, 25,  1, 12, 21, 23, 13, 21, 25,
          1, 25, 14, 11,  1, 25, 26, 20, 11],
        [ 4,  1, 28, 23, 21, 25, 11,  1,  7,  1, 18, 21, 27, 11,  1, 24, 21, 20,
         13,  1, 25, 21,  1, 19, 29,  1,  8, 23, 21, 17, 11, 20,  1, 19, 15,  9,
         23, 21, 28,  7, 27, 11,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
          0,  0,  0,  0,  0,  0,  0,  0,  0],
        [ 6, 14, 11,  1, 24, 15, 18, 11, 20,  9, 11,  1, 15, 20,  1, 25, 14, 15,
         24,  1, 11, 19, 22, 25, 29,  1, 23, 21, 21, 19,  1, 24, 25, 15, 18, 18,
          1, 28, 14, 15, 24, 22, 11, 23, 24,  1, 29, 21, 26, 23,  1, 20,  7, 19,
         11,  0,  0,  0,  0,  0,  0,  0,  0],
        [ 4,  1, 17, 11, 22, 25,  1, 29, 21, 26, 23,  1, 18, 11, 25, 25, 11, 23,
          1, 12, 21, 18, 10, 11, 10,  1,  8, 29,  1,

## Section 8: Build a Simple Classification Head

In Chapter 6, the generation head is replaced with a classification head.

In a full GPT-style model:

```text
Transformer output
        ↓
Classification head
        ↓
Class logits
```

For this first notebook, we will use a small classifier that contains:

1. Token embedding layer
2. Mean pooling across tokens
3. Linear classification head

This is not a full GPT classifier, but it demonstrates the same core idea: converting token sequences into class logits.

In [15]:
import torch.nn as nn


class TinyLyricsClassifier(nn.Module):
    def __init__(self, vocab_size, emb_dim, num_classes):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, emb_dim)
        self.classifier = nn.Linear(emb_dim, num_classes)

    def forward(self, input_ids):
        # input_ids shape:
        # (batch_size, num_tokens)

        embeddings = self.embedding(input_ids)

        # embeddings shape:
        # (batch_size, num_tokens, emb_dim)

        # Average embeddings across the token dimension.
        pooled = embeddings.mean(dim=1)

        logits = self.classifier(pooled)

        return logits

## Section 9: Test the Classifier Output Shape

For classification with two labels, the model should produce two logits per input example.

If the batch size is 4, the output should be:

```text
(batch_size, num_classes)
```

or:

```text
(4, 2)
```

In [16]:
vocab_size = len(tokenizer.chars)
emb_dim = 32
num_classes = 2

model = TinyLyricsClassifier(
    vocab_size=vocab_size,
    emb_dim=emb_dim,
    num_classes=num_classes
).to(device)

input_batch, label_batch = next(iter(train_loader))
input_batch = input_batch.to(device)

logits = model(input_batch)

print(logits)
print(logits.shape)

tensor([[ 0.4967, -0.0013],
        [ 0.5108, -0.0385],
        [ 0.4006,  0.0452],
        [ 0.3331,  0.0533]], grad_fn=<AddmmBackward0>)
torch.Size([4, 2])


## Section 10: Calculate Classification Loss

For classification, we use cross-entropy loss.

The model produces logits shaped like:

```text
(batch_size, num_classes)
```

The labels are shaped like:

```text
(batch_size)
```

In [17]:
import torch.nn.functional as F

input_batch, label_batch = next(iter(train_loader))

input_batch = input_batch.to(device)
label_batch = label_batch.to(device)

logits = model(input_batch)

loss = F.cross_entropy(logits, label_batch)

print(loss)

tensor(0.8211, grad_fn=<NllLossBackward0>)


## Section 11: Calculate Accuracy

Accuracy measures the percentage of examples classified correctly.

TODO:

Complete the `calculate_accuracy` function.

In [22]:
def calculate_accuracy(data_loader, model, device):
    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():
        for input_batch, label_batch in data_loader:
            input_batch = input_batch.to(device)
            label_batch = label_batch.to(device)

            logits = model(input_batch)

            predicted_labels = torch.argmax(logits, dim=1)

            correct += (predicted_labels == label_batch).sum().item()
            total += label_batch.size(0)

    return correct / total


print(calculate_accuracy(val_loader, model, device))

0.0


## Section 12: Train the Classifier

This is a small supervised fine-tuning loop.

Notice the difference from language model pretraining:

- Pretraining target: next token
- Classification target: class label

In [23]:
optimizer = torch.optim.AdamW(model.parameters(), lr=0.01)

num_epochs = 10

for epoch in range(num_epochs):
    model.train()

    total_loss = 0.0

    for input_batch, label_batch in train_loader:
        input_batch = input_batch.to(device)
        label_batch = label_batch.to(device)

        optimizer.zero_grad()

        logits = model(input_batch)
        loss = F.cross_entropy(logits, label_batch)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)

    train_accuracy = calculate_accuracy(train_loader, model, device)
    val_accuracy = calculate_accuracy(val_loader, model, device)

    print(
        f"Epoch {epoch + 1}: loss = {avg_loss:.4f} | "
        f"train accuracy = {train_accuracy:.2%} | "
        f"validation accuracy = {val_accuracy:.2%}"
    )

Epoch 1: loss = 0.3359 | train accuracy = 100.00% | validation accuracy = 50.00%
Epoch 2: loss = 0.3085 | train accuracy = 100.00% | validation accuracy = 0.00%
Epoch 3: loss = 0.2721 | train accuracy = 100.00% | validation accuracy = 0.00%
Epoch 4: loss = 0.2464 | train accuracy = 100.00% | validation accuracy = 0.00%
Epoch 5: loss = 0.2223 | train accuracy = 100.00% | validation accuracy = 0.00%
Epoch 6: loss = 0.1996 | train accuracy = 100.00% | validation accuracy = 0.00%
Epoch 7: loss = 0.1792 | train accuracy = 100.00% | validation accuracy = 0.00%
Epoch 8: loss = 0.1623 | train accuracy = 100.00% | validation accuracy = 0.00%
Epoch 9: loss = 0.1436 | train accuracy = 100.00% | validation accuracy = 0.00%
Epoch 10: loss = 0.1293 | train accuracy = 100.00% | validation accuracy = 0.00%


## Section 13: Evaluate on Test Data

The test set should only be used after training decisions are finished.

TODO:

1. Calculate test accuracy.
2. Try a few new lyric-like examples.
3. Decide whether the classifier is actually useful.

In [24]:
test_accuracy = calculate_accuracy(test_loader, model, device)
print(f"Test accuracy: {test_accuracy:.2%}")

Test accuracy: 50.00%


## Section 14: Classify New Text

Now test the classifier on new examples.

TODO:

Complete the helper function below.

In [ ]:
def classify_text(text, model, tokenizer, max_length, device):
    model.eval()

    encoded = tokenizer.encode(text)

    encoded = encoded[:max_length]

    encoded = encoded + [0] * (max_length - len(encoded))

    input_tensor = torch.tensor(
        encoded,
        dtype=torch.long
    ).unsqueeze(0).to(device)

    with torch.no_grad():
        logits = model(input_tensor)

    predicted_label = torch.argmax(logits, dim=1).item()

    return predicted_label


examples = [
    "My sandwich started singing opera in the shower",
    "I still remember the night you walked away",
]

for example in examples:
    label = classify_text(
        example,
        model,
        tokenizer,
        train_dataset.max_length,
        device
    )

    label_name = "silly/comedic" if label == 1 else "serious"
    print(example, "=>", label_name)

My sandwich started singing opera in the shower => silly/comedic
I still remember the night you walked away => silly/comedic


: 

## Section 15: Reflection Questions

Answer these questions in your submitted Word document.

1. How is classification different from text generation?
2. Why does classification require labeled data?
3. Why do we need to pad examples in a classification dataset?
4. Why does the classification head output two values instead of vocabulary-size values?
5. How could a lyric classifier support the future Weird AI parody assistant?
6. What limitations did you notice with this tiny dataset?
7. If you wanted this classifier to be more useful, what data would you need?

## Optional Extension: Connect This to the Full Weird AI Model

In this notebook, we used a tiny classifier for clarity.

In a full version, you could:

1. Load the pretrained Weird AI transformer.
2. Freeze most of its weights.
3. Replace the generation head with a classification head.
4. Fine-tune the classifier on labeled lyric data.

This is the same high-level strategy used in Chapter 6.